# ClinGuard-DiLoCo: Kaggle session 1 (environment, access, profiling)

Gates G0 and G3 (stage S0). Run **top to bottom**, once, on Tasbid's account.

**Before starting** (right-hand panel):
1. *Accelerator*: **GPU T4 x2**. *Internet*: **on**.
2. *Add Input*: dataset **raddar/chest-xrays-indiana-university**.
3. *Add-ons -> Secrets*: `HF_TOKEN` (write token for the team's private Hugging Face repo), attached to this notebook.
4. Keep this notebook **private**. Its outputs include generated report text (licence CC BY-NC-ND).

Fill in `HF_REPO` below. Nothing else needs editing.

In [ ]:
HF_REPO = "ORG_NAME/REPO_NAME"   # <- the private model repo, e.g. "my-org/clinguard-runs"
GIT_REF = "main"                 # session 1 only; training runs pin an exact commit

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1

## 2. Libraries (pinned to the versions the CPU tests passed with; Kaggle's own torch is kept)

In [ ]:
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "cuda", torch.version.cuda, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

## 3. Data: link the Kaggle dataset where the code expects it, then verify it is byte-identical to the frozen build

In [ ]:
import glob, os
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
print(src, os.listdir("data/raw/iu_xray"))
!python scripts/prepare_data.py --raw-dir data/raw/iu_xray --images-dir data/raw/iu_xray/images/images_normalized --check

## 4. Test suite (CPU parts plus the tiny-model pipeline tests on real images)

In [ ]:
!python -m pytest -q -p no:warnings 2>&1 | tail -5

## 5. Hugging Face write access (assumption A9): write and delete a small file

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi()
me = api.whoami()["name"]
api.upload_file(path_or_fileobj=f"write check from {me}".encode(), path_in_repo=f"checks/{me}.txt",
                repo_id=HF_REPO, repo_type="model", commit_message=f"A9 write check ({me})")
api.delete_file(f"checks/{me}.txt", repo_id=HF_REPO, repo_type="model", commit_message=f"A9 cleanup ({me})")
print("A9 OK for", me, "->", HF_REPO, "| private:", api.repo_info(HF_REPO).private)

## 6. Profile the real model on one T4 (about 20-30 minutes)

Loads Qwen2.5-VL-3B in 4-bit, creates the shared LoRA A file, measures memory and seconds per optimizer step
at 448 and 336 pixels with micro-batches 1, 2 and 4, and times report generation.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/profile_model.py --raw-dir data/raw/iu_xray     --images-dir data/raw/iu_xray/images/images_normalized --out /kaggle/working/profile

## 7. Upload the results to the private HF repo (LoRA A file, profile, zero-shot samples)

In [ ]:
import json
prof = json.load(open("/kaggle/working/profile/profile.json"))
api.upload_file(path_or_fileobj="/kaggle/working/profile/lora_A.safetensors",
                path_in_repo="shared/lora_A_r16_seed2026.safetensors", repo_id=HF_REPO,
                commit_message=f"LoRA A (sha256 {prof['lora_a_sha256'][:12]})")
api.upload_folder(folder_path="/kaggle/working/profile", path_in_repo="profiles/session1", repo_id=HF_REPO,
                  ignore_patterns=["*.safetensors"], commit_message="Session 1 profile")
print(json.dumps({k: prof[k] for k in ("n_phi", "lora_a_sha256", "memory_after_load_gb", "load_sec")}, indent=1))
for r in prof["training"]: print(r)
for g in prof["generation"]: print(g)

## 8. Metric packages (assumption A7). Run last: these installs may change other packages.

In [ ]:
!pip install -q radgraph f1chexbert 2>&1 | tail -3
for mod in ("radgraph", "f1chexbert"):
    try:
        __import__(mod); print(mod, "imports OK")
    except Exception as e:
        print(mod, "FAILED:", repr(e)[:300])

**Send Tasbid / Claude:** the printed output of cells 3, 4, 5, 7 and 8 (not the zero-shot texts).